<a href="https://colab.research.google.com/github/LennartRedlich/Capstone-Project-/blob/Anh/src/notebooks/Baseline%20Seniority.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline: Seniority Prediction

In this notebook, we implement a simple baseline for predicting job seniority based solely on the job title.

In [4]:
import json
from pathlib import Path
import pandas as pd

In [7]:
DATA_PATH = Path.cwd()
DATA_PATH

PosixPath('/content')

In [8]:
with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

len(cvs)

609

In [9]:
jobs = []
for cv in cvs:
    for job in cv:
        jobs.append(job)

df = pd.DataFrame(jobs)
df.head(15)


,organization,linkedin,position,startDate,endDate,status,department,seniority
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
5,Nagel Car Group,,Buchhalterin,2000-05,2019-06,INACTIVE,Other,Professional
6,Computer Solutions,https://www.linkedin.com/company/computer-solu...,Solutions Architect,2024-03,None,ACTIVE,Information Technology,Professional
7,Computer Solutions,https://www.linkedin.com/company/computer-solu...,Senior Network Engineer,2019-07,2024-03,INACTIVE,Information Technology,Senior
8,Texas A&M University-Corpus Christi,,Manager of Network Services,2017-02,2019-07,INACTIVE,Information Technology,Professional
9,Texas A&M University-Corpus Christi,,Infrastructure Administrator II,2015-06,2017-02,INACTIVE,Information Technology,Professional


In [10]:
df_active = df[df["status"] == "ACTIVE"].copy()
df_active.shape

(623, 8)

In [11]:
df_active["seniority"].value_counts()

,count
seniority,
Professional,216
Management,192
Lead,125
Senior,44
Director,34
Junior,12


# Baseline Idea


Firstly, we create a dictionary including the label of seniority as the key, and the relevent predefined title as the value from provided file "seniority-v2.csv".

In [16]:
df_seniority = pd.read_csv("seniority-v2.csv")

seniority_dict = (
    df_seniority
    .groupby("label")["text"]
    .apply(list)
    .to_dict()
)

Create a function to match the position with the label of seniority from the dictionary

In [42]:
def predict_seniority(sen, seniority_dict):
    sen = sen.lower()

    for label, texts in seniority_dict.items():
        for t in texts:
            if t.lower() in sen:
                return label

    return "Professional"

Apply to the data set to predict the seniority

In [43]:
predictions = []

for sen in df_active["position"]:
    pred = predict_seniority(sen, seniority_dict)
    predictions.append(pred)

df_active["predicted_seniority"] = predictions

In [44]:
df_active["predicted_seniority"].value_counts()

,count
predicted_seniority,
Professional,240
Management,120
Senior,119
Lead,73
Director,52
Junior,19


In [45]:
df_active['seniority'].value_counts()

,count
seniority,
Professional,216
Management,192
Lead,125
Senior,44
Director,34
Junior,12


In [46]:
accuracy = (
    df_active["seniority"] == df_active["predicted_seniority"]
).mean()

accuracy

np.float64(0.5906902086677368)